In [12]:
import pandas as pd

In [33]:
#menu dataset
df = pd.read_csv(r"C:\Users\liann\Downloads\training_data.csv")
df

,Food Product,Main Ingredient,Sweetener,Fat/Oil,Seasoning,Allergens,Prediction,source
0,Almond Cookies,Almonds,Sugar,Butter,Flour,"Dairy, Tree Nuts, Wheat",Contains,original
1,Cheddar Cheese,Cheese,NaN,NaN,Salt,Dairy,Contains,original
2,Ranch Dressing,Buttermilk,Sugar,Vegetable oil,"Garlic, herbs",Dairy,Contains,original
3,Caramel Popcorn,Popcorn,Sugar,Butter,Salt,Dairy,Contains,original
4,Caesar Salad,Romaine lettuce,NaN,Olive oil,Parmesan cheese,Dairy,Contains,original
...,...,...,...,...,...,...,...,...
15708,60 Piece Wings Platter,Specialty Items - Available 10:30AM to Close,NaN,NaN,"60 Fresh, never frozen wing portions lightly b...",Wheat,Contains,uber_eats
15709,Grilled Chicken Sandwich Combo - 10:30AM to Close,Favorite Combos - Available 10:30AM to Close,NaN,NaN,"A tender grilled chicken breast, lettuce, toma...",Wheat,Contains,uber_eats
15710,Macaroni & Cheese - 10:30AM to Close,Fixins,NaN,NaN,Creamy and cheesy Mac and Cheese!\r\n,"Dairy, Wheat",Contains,uber_eats
15711,Bourbon Street Chicken & Shrimp,Picked for you,NaN,NaN,Let the good times roll with Cajun-seasoned ch...,"Garlic, Shellfish",Contains,uber_eats


In [30]:
database = pd.read_csv(r"C:\Users\liann\Downloads\allergen_database_types.csv")
database

,group_type,group_name,ingredient_type,ingredient,match_type,confidence_score,notes
0,allergen,milk,common_ingredient,milk,exact_match,0.95,NaN
1,allergen,milk,common_ingredient,whole milk,exact_match,0.95,NaN
2,allergen,milk,common_ingredient,skim milk,exact_match,0.95,NaN
3,allergen,milk,common_ingredient,cream,exact_match,0.95,NaN
4,allergen,milk,common_ingredient,butter,exact_match,0.95,NaN
...,...,...,...,...,...,...,...
462,allergen,corn,hidden_ingredient,masa,hidden_ingredient,0.85,Corn dough
463,allergen,corn,hidden_ingredient,hominy,hidden_ingredient,0.85,Corn product
464,allergen,corn,hidden_ingredient,cornstarch,hidden_ingredient,0.85,Corn starch
465,allergen,corn,dish_clue,tortilla chips,dish_clue,0.65,Often corn-based


In [36]:
import re
import pandas as pd
from sklearn.model_selection import train_test_split

#split into test/val/train: 70%, 15%, 15%

train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    random_state=42,
    shuffle=True
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    shuffle=True
)

X_train = train_df.drop(columns=["Allergens"]).reset_index(drop=True) #separting inputs and dfs with allergen answers
y_train = train_df["Allergens"].reset_index(drop=True)

X_val = val_df.drop(columns=["Allergens"]).reset_index(drop=True)
y_val = val_df["Allergens"].reset_index(drop=True)

X_test = test_df.drop(columns=["Allergens"]).reset_index(drop=True)
y_test = test_df["Allergens"].reset_index(drop=True)

#standardize data

def clean_text(text):
    text = str(text).lower()
    text = text.replace("-", " ")
    text = text.replace("_", " ")
    text = re.sub(r"[^a-zA-Z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def normalize_allergen(text):
    text = clean_text(text)

    synonym_map = {
        "milk": "dairy",
        "dairy": "dairy",

        "wheat": "wheat",
        "gluten": "wheat",

        "egg": "eggs",
        "eggs": "eggs",

        "soybean": "soy",
        "soybeans": "soy",
        "soy": "soy",

        "peanut": "peanuts",
        "peanuts": "peanuts",

        "tree nuts": "tree nuts",
        "tree nut": "tree nuts",
        "nuts": "tree nuts",

        "shellfish": "shellfish",
        "shrimp": "shellfish",
        "crab": "shellfish",
        "lobster": "shellfish",

        "fish": "fish",

        "sesame": "sesame",
        "mustard": "mustard",
        "celery": "celery",
        "garlic": "garlic",
        "onion": "onion",
        "corn": "corn"
    }

    return synonym_map.get(text, text)

def combine_row_text(row):
    return " ".join(
        clean_text(value)
        for value in row.values
        if pd.notna(value)
    )

def split_allergens(value):
    if pd.isna(value):
        return []

    return sorted([
        normalize_allergen(a)
        for a in str(value).split(",")
        if a.strip()
    ])

#standardize database for matching

database["ingredient_clean"] = database["ingredient"].apply(clean_text)
database["group_name_clean"] = database["group_name"].apply(normalize_allergen)

#rule based prediction

def predict_rule_based(row, trust_threshold=0.80):  #use 0.80 as default threshold, rest goes to ML model
    """
    takes one food item row and checks whether any database ingredient appears in it
    """
    
    text = combine_row_text(row) #turns all columns in menu dataset into one searchable string

    predicted_allergens = set()
    matches = []

    for _, db_row in database.iterrows():
        ingredient = db_row["ingredient_clean"]

        if ingredient == "":
            continue

        allergen = normalize_allergen(db_row["group_name_clean"])
        ingredient_type = db_row["ingredient_type"]
        match_type = db_row["ingredient_type"]
        confidence = float(db_row["confidence_score"])

        pattern = r"\b" + re.escape(ingredient) + r"\b" #so it only extracts the exact word, not part of one

        if re.search(pattern, text): #adds allergen predictions
            predicted_allergens.add(allergen)

            matches.append({ #stores why the allergen was predicted; ex. allergen: wheat
                "allergen": allergen,
                "matched_ingredient": ingredient,
                "ingredient_type": ingredient_type,
                "match_type": match_type,
                "confidence": confidence
            })

    if len(matches) == 0: #if no matches found send to ML model
        return {
            "predicted_allergens": [],
            "matches": [],
            "max_confidence": 0.0,
            "decision": "send_to_ml_or_review"
        }

    max_confidence = max(match["confidence"] for match in matches)

    if max_confidence >= trust_threshold: #deciding whether or not to trust rule-based
        decision = "trust_rule_based"
    else:
        decision = "send_to_ml_or_review"

    return {
        "predicted_allergens": sorted(predicted_allergens),
        "matches": matches,
        "max_confidence": max_confidence,
        "decision": decision
    }

#test on validation set

validation_results = []

for i in range(len(X_val)):
    row = X_val.iloc[i]
    actual = split_allergens(y_val.iloc[i]) #get the correct answer to make answer key

    result = predict_rule_based(row)

    predicted_set = set(result["predicted_allergens"])
    actual_set = set(actual)

    #compares answer key against predictions
    correct_matches = predicted_set & actual_set
    extra_predictions = predicted_set - actual_set #predicted incorrectly
    missed_allergens = actual_set - predicted_set #allergens undetected

    if len(predicted_set) > 0: #how many predictions were correct?
        precision = len(correct_matches) / len(predicted_set)
    else:
        precision = 0

    if len(actual_set) > 0: #how many real allergens did we catch?
        recall = len(correct_matches) / len(actual_set)
    else:
        recall = 1

    if precision + recall > 0: #performance score
        f1 = 2 * precision * recall / (precision + recall)
    else:
        f1 = 0

    validation_results.append({
        "food_product": row.get("Food Product", ""),
        "actual_allergens": actual,
        "predicted_allergens": result["predicted_allergens"],
        "max_confidence": result["max_confidence"],
        "decision": result["decision"],
        "exact_match_correct": predicted_set == actual_set,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "extra_predictions": sorted(extra_predictions),
        "missed_allergens": sorted(missed_allergens),
        "matches": result["matches"]
    })

validation_results_df = pd.DataFrame(validation_results)

print("Validation exact row-level accuracy:")
print(validation_results_df["exact_match_correct"].mean())

print("\nValidation average precision:")
print(validation_results_df["precision"].mean())

print("\nValidation average recall:")
print(validation_results_df["recall"].mean())

print("\nValidation average F1:")
print(validation_results_df["f1"].mean())

validation_results_df.head()

#break rule match into own rows

match_rows = []

for _, row in validation_results_df.iterrows():
    for match in row["matches"]:
        match_rows.append({
            "food_product": row["food_product"],
            "actual_allergens": row["actual_allergens"],
            "predicted_allergen": match["allergen"],
            "matched_ingredient": match["matched_ingredient"],
            "ingredient_type": match["ingredient_type"],
            "match_type": match["match_type"],
            "confidence": match["confidence"],
            "is_correct": match["allergen"] in row["actual_allergens"]
        })

match_results_df = pd.DataFrame(match_rows)

match_results_df.head()

#check reliability by match type

match_type_reliability = (
    match_results_df
    .groupby("ingredient_type")
    .agg(
        total_matches=("is_correct", "count"),
        correct_matches=("is_correct", "sum"),
        accuracy=("is_correct", "mean"),
        avg_confidence=("confidence", "mean")
    )
    .reset_index()
)

print("\nReliability by match type:")
print(match_type_reliability)

#check which confidence score gives best results

thresholds = [0.50, 0.60, 0.65, 0.70, 0.75, 0.80, 0.85, 0.90, 0.95]

threshold_results = []

for threshold in thresholds:
    trusted = match_results_df[match_results_df["confidence"] >= threshold]

    if len(trusted) == 0:
        continue

    threshold_results.append({
        "threshold": threshold,
        "num_trusted_matches": len(trusted),
        "accuracy_when_trusted": trusted["is_correct"].mean()
    })

threshold_results_df = pd.DataFrame(threshold_results)

print("\nThreshold results:")
print(threshold_results_df)

#choose lowest threshold with at least 90% accuracy
reliable_thresholds = threshold_results_df[
    threshold_results_df["accuracy_when_trusted"] >= 0.90
]

if len(reliable_thresholds) > 0:
    TRUST_THRESHOLD = reliable_thresholds["threshold"].min()
else:
    TRUST_THRESHOLD = 0.80

print("\nChosen trust threshold:", TRUST_THRESHOLD)

#log mistakes

#mistake type 1: predicted allergen that was not actually in the answer
false_positive_log = match_results_df[
    match_results_df["is_correct"] == False
].copy()

false_positive_log["issue"] = "rule predicted allergen but actual answer did not include it"
false_positive_log["suggested_action"] = "lower confidence, change match_type, or remove/change ingredient rule"

false_positive_log.to_csv("rule_based_false_positives.csv", index=False)

#mistake type 2: missed allergens that the rule system failed to catch
missed_rows = []

for _, row in validation_results_df.iterrows():
    for missed in row["missed_allergens"]:
        missed_rows.append({
            "food_product": row["food_product"],
            "actual_allergens": row["actual_allergens"],
            "predicted_allergens": row["predicted_allergens"],
            "missed_allergen": missed,
            "issue": "rule system missed allergen",
            "suggested_action": "add synonym, hidden ingredient, or dish clue to allergen database"
        })

missed_allergens_log = pd.DataFrame(missed_rows)

missed_allergens_log.to_csv("rule_based_missed_allergens.csv", index=False)

print("\nSaved mistake logs:")
print("rule_based_false_positives.csv")
print("rule_based_missed_allergens.csv")

#view mistakes

print("\nFalse positives:")
display(false_positive_log.head(5))

print("\nMissed allergens:")
display(missed_allergens_log.head(5))

Validation exact row-level accuracy:
0.10818837505303351

Validation average precision:
0.32921664673892076

Validation average recall:
0.9922924621694242

Validation average F1:
0.45259848575504075

Reliability by match type:
          ingredient_type  total_matches  correct_matches  accuracy  \
0  allowed_with_condition           5370                0  0.000000   
1                   avoid           8655                0  0.000000   
2       common_ingredient           4986             4961  0.994986   
3       hidden_ingredient             55               53  0.963636   

   avg_confidence  
0            0.60  
1            0.60  
2            0.95  
3            0.85  

Threshold results:
   threshold  num_trusted_matches  accuracy_when_trusted
0       0.50                19066               0.262981
1       0.60                19066               0.262981
2       0.65                 5041               0.994644
3       0.70                 5041               0.994644
4       0.75

,food_product,actual_allergens,predicted_allergen,matched_ingredient,ingredient_type,match_type,confidence,is_correct,issue,suggested_action
0,Classic Chicken Sandwich Box,[wheat],vegan,chicken,avoid,avoid,0.6,False,rule predicted allergen but actual answer did ...,"lower confidence, change match_type, or remove..."
1,Classic Chicken Sandwich Box,[wheat],vegetarian,chicken,avoid,avoid,0.6,False,rule predicted allergen but actual answer did ...,"lower confidence, change match_type, or remove..."
2,Classic Chicken Sandwich Box,[wheat],pescatarian,chicken,avoid,avoid,0.6,False,rule predicted allergen but actual answer did ...,"lower confidence, change match_type, or remove..."
3,Classic Chicken Sandwich Box,[wheat],halal,chicken,allowed_with_condition,allowed_with_condition,0.6,False,rule predicted allergen but actual answer did ...,"lower confidence, change match_type, or remove..."
4,Classic Chicken Sandwich Box,[wheat],kosher,chicken,allowed_with_condition,allowed_with_condition,0.6,False,rule predicted allergen but actual answer did ...,"lower confidence, change match_type, or remove..."



Missed allergens:


,food_product,actual_allergens,predicted_allergens,missed_allergen,issue,suggested_action
0,Almond Cookies,"[dairy, tree nuts, wheat]","[dairy, dairy free, nut free, tree nuts, vegan...",wheat,rule system missed allergen,"add synonym, hidden ingredient, or dish clue t..."
1,BBQ Ranch Chicken Salad,"[corn, legumes]","[corn, gluten free, halal, kosher, legume, pes...",legumes,rule system missed allergen,"add synonym, hidden ingredient, or dish clue t..."
2,Shrimp and Black Bean Sauce,"[legumes, shellfish]","[gluten free, halal, kosher, legume, pescatari...",legumes,rule system missed allergen,"add synonym, hidden ingredient, or dish clue t..."
3,Chicken Caesar Salad,"[dairy, wheat]","[fish, halal, kosher, pescatarian, vegan, vege...",dairy,rule system missed allergen,"add synonym, hidden ingredient, or dish clue t..."
4,Chicken Caesar Salad,"[dairy, wheat]","[fish, halal, kosher, pescatarian, vegan, vege...",wheat,rule system missed allergen,"add synonym, hidden ingredient, or dish clue t..."


In [ ]:
#need to account for "avoid_with_condtion" and "avoid" if time, these are for food restrictions, not allergens
#use 0.65 in predict_rule_based) now